# LSC50 — extracción de landmarks en Colab

Procesa los 1000 clips RGB de LSC50 y deja un `.jsonl` con el formato de
`docs/06-data/model-input-format.md`: 30 frames × 128 valores, normalizados
al centro y la distancia de los hombros.

**Por qué acá y no en local:** Figshare entrega a ~120 KB/s desde una máquina
de casa (14 h para 6,25 GB) y la extracción tarda ~2 h en un núcleo. Colab baja
el ZIP en minutos y reparte el trabajo entre sus CPU.

**Antes de correr:** Entorno de ejecución → Cambiar tipo → CPU alta memoria.
La GPU no ayuda: MediaPipe corre en CPU y el cuello de botella es decodificar video.

## 1. Dependencias

In [ ]:
!pip install -q mediapipe opencv-python-headless

import multiprocessing, os
print('núcleos disponibles:', multiprocessing.cpu_count())

## 2. El extractor

Se trae desde el repo para que sea **el mismo código** que se validó en local.
Si no hay acceso al repo, se sube `extract_landmarks.py` a mano con el panel
de archivos de Colab.

In [ ]:
REPO = 'https://github.com/mauro-1508/backendTS.git'
RAMA = 'feat/backend-dominios-ia'

import os
if not os.path.exists('extract_landmarks.py'):
    !git clone -q --branch {RAMA} --depth 1 {REPO} repo
    !cp repo/src/domains/ia/training/extraction/extract_landmarks.py .

import extract_landmarks as ex
print('formato:', ex.SEQ_LEN, 'frames x', ex.FEATURE_DIM, 'valores')

## 3. Descarga de LSC50

Solo hace falta `VIDEOS.zip` (6,25 GB). De ahí se extrae únicamente
`COLOR_BODY`, que son los 1000 clips RGB de cuerpo completo; las otras tres
modalidades (cara, profundidad e infrarrojo) no entran en el plan.

In [ ]:
VIDEOS_URL = 'https://ndownloader.figshare.com/files/50118645'  # LSC50 VIDEOS.zip

!wget -q --show-progress -O videos.zip {VIDEOS_URL}
!unzip -q -o videos.zip 'VIDEOS/COLOR_BODY/*' -d lsc50
!ls lsc50/VIDEOS/COLOR_BODY | wc -l

## 4. Extracción en paralelo

Cada proceso abre sus propios modelos de MediaPipe: los objetos de Tasks no se
pueden compartir entre procesos.

In [ ]:
import json, time
from concurrent.futures import ProcessPoolExecutor

VIDEOS_DIR = 'lsc50/VIDEOS/COLOR_BODY'
# A Drive directo: si Colab se desconecta, el disco local se borra.
from google.colab import drive
drive.mount('/content/drive')
SALIDA = '/content/drive/MyDrive/traduce_senas/datasets/lsc50.jsonl'
os.makedirs(os.path.dirname(SALIDA), exist_ok=True)

hand_model = ex.ensure_model(ex.HAND_MODEL_URL, 'models')
pose_model = ex.ensure_model(ex.POSE_MODEL_URL, 'models')

def procesar(nombre):
    try:
        return ex.extract_clip(os.path.join(VIDEOS_DIR, nombre), hand_model, pose_model)
    except Exception as err:  # un clip roto no debe tumbar la corrida entera
        return {'error': str(err), 'source_clip': nombre}

videos = sorted(v for v in os.listdir(VIDEOS_DIR) if v.lower().endswith('.avi'))
print(len(videos), 'clips por procesar')

inicio = time.time()
ok = fallidos = 0
with open(SALIDA, 'w', encoding='utf-8') as fh, ProcessPoolExecutor() as pool:
    for i, rec in enumerate(pool.map(procesar, videos, chunksize=4), 1):
        if rec is None or 'error' in rec:
            fallidos += 1
        else:
            fh.write(json.dumps(rec, ensure_ascii=False) + '\n')
            ok += 1
        if i % 50 == 0:
            transcurrido = time.time() - inicio
            print(f'{i}/{len(videos)} | {ok} ok, {fallidos} fallidos | '
                  f'{transcurrido/i:.1f} s/clip | faltan {(len(videos)-i)*transcurrido/i/60:.0f} min')

print(f'\nlisto: {ok} clips en {SALIDA} ({fallidos} fallidos) en {(time.time()-inicio)/60:.0f} min')

## 5. Revisión antes de guardar

In [ ]:
import numpy as np

recs = [json.loads(l) for l in open(SALIDA, encoding='utf-8')]
X = np.array([r['frames'] for r in recs], dtype=np.float32)
coords = X[:, :, :126]

print('tensor:', X.shape, '(esperado: n x 30 x 128)')
print('señas:', len({r['sign_id'] for r in recs}), '| señantes:', sorted({r['signer'] for r in recs}))
print('banderas de presencia:', np.unique(X[:, :, 126:]), '(deben ser solo 0 y 1)')
print('manos vistas: %.0f%%' % (100 * X[:, :, 126:].mean()))

nz = coords[np.abs(coords) > 1e-9]
print('coordenadas: media %+.2f, desviación %.2f, rango %+.2f a %+.2f' % (
    nz.mean(), nz.std(), nz.min(), nz.max()))

# La muñeca derecha debe caer a la izquierda de la imagen (x negativo) y al revés.
wr, wl = X[:, :, 0], X[:, :, 63]
print('x muñeca derecha %+.2f | izquierda %+.2f (deben tener signos opuestos)' % (
    np.nanmean(wr[np.abs(wr) > 1e-9]), np.nanmean(wl[np.abs(wl) > 1e-9])))

por_senante = {}
for r in recs:
    por_senante[r['signer']] = por_senante.get(r['signer'], 0) + 1
print('clips por señante:', por_senante)

## 6. Guardar en Drive

El `.jsonl` pesa unos 200 MB. Va a Drive para que el notebook de entrenamiento
lo tome sin repetir la extracción.

In [ ]:
# Ya quedo en Drive.
!ls -lh /content/drive/MyDrive/traduce_senas/datasets/
